# 중간고사 대비 · 데이터 전처리와 모델 학습
기존 모델에서 사용하는 Iris, Pima, Ames, ETTh1, IRB2400 데이터만 사용합니다. Wine은 제외했습니다.
Fashion-MNIST는 포함하지 않습니다. Iris는 이미지 대신 샘플별 특성값을 2×5 막대그래프로 표시합니다.

각 문제는 **문제 설명 셀 1개 + 코드 셀 1개**입니다. 공통 준비 후 위에서부터 풀어 주세요.
선행 문제가 표시된 문제는 앞 문제에서 만든 변수를 사용합니다. 데이터셋마다 변수 앞 이름을 달리했습니다.

CSV 파일을 Drive의 `MyDrive/Colab Notebooks/data/`에 준비하세요.
로컬에서는 각 데이터 문제의 `FILE_PATH`를 실제 CSV 위치로 수정하세요.
`fit`은 훈련에만, 검증은 모델 선택에, 테스트는 최종 평가에만 사용합니다.
정확도 기준을 맞추는 시험이 아니라 전처리·shape·규제·평가 흐름을 연습하는 자료입니다.

**문제용:** TODO를 작성하세요. 각 문제의 코드는 바로 아래 한 셀에 작성합니다.


## 공통 준비 1. 라이브러리와 설정
필요하면 `%pip install -r requirements.txt`를 먼저 실행하세요.


In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.optimizers import AdamW
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    MinMaxScaler,
    StandardScaler,
    LabelEncoder,
    OneHotEncoder,
)
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    ConfusionMatrixDisplay,
)
import joblib

# 정답의 저장 출력은 빠른 점검 모드에서 직접 실행한 결과입니다.
# 전체 학습은 False로 바꿔 공통 준비부터 다시 실행하세요.
QUICK_RUN = False
EPOCHS = 2 if QUICK_RUN else 60
SEED = 42
keras.utils.set_random_seed(SEED)
np.set_printoptions(precision=4, suppress=True)
SAVE_DIR = Path("midterm_artifacts")
SAVE_DIR.mkdir(exist_ok=True)


def limit_for_quick(Xtr, Xva, Xte, ytr, yva, yte):
    """분할 이후에만 점검용 샘플 수를 줄입니다. 전체 연습에서는 그대로 반환합니다."""
    print("Full train/validation/test:", len(ytr), len(yva), len(yte))
    if QUICK_RUN:
        Xtr, ytr = Xtr[:1024], ytr[:1024]
        Xva, yva = Xva[:256], yva[:256]
        Xte, yte = Xte[:256], yte[:256]
    print("Used train/validation/test:", len(ytr), len(yva), len(yte))
    return Xtr, Xva, Xte, ytr, yva, yte


print("TensorFlow:", tf.__version__, "| Quick run:", QUICK_RUN, "| Epochs:", EPOCHS)


## 공통 준비 2. Google Drive 연결


In [ ]:
try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print(
        "Google Colab이 아닙니다. 각 데이터 문제의 FILE_PATH를 로컬 경로로 수정하세요."
    )


## 문제 1. Iris 불러오기와 검증 데이터 분리

**선행:** 공통 준비

1. Drive의 iris3.csv를 읽고 입력 4개와 species를 분리하세요.
2. 클래스 비율을 유지하면서 훈련/검증/테스트를 60/20/20으로 나누세요.
3. 분할 크기와 훈련 클래스별 개수를 출력하세요.


In [ ]:
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/iris3.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 iris3.csv 위치로 수정하세요."
    )

# 각 문제에서 사용할 원본 CSV를 읽습니다.
iris_df = pd.read_csv(FILE_PATH)

# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 2. Iris 0~1 정규화와 라벨 인코딩

**선행:** 문제 1

1. MinMaxScaler는 훈련 입력에만 fit하고 검증·테스트에는 transform하세요.
2. species를 정수로 인코딩한 뒤 클래스 3개의 one-hot 정답을 만드세요.
3. 훈련 입력의 최솟값·최댓값과 변환 후 shape를 출력하세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 3. 훈련 데이터 앞 10개를 2×5로 시각화

**선행:** 문제 2

1. 정규화한 Iris 훈련 샘플 앞 10개를 2행×5열로 배치하세요.
2. 각 샘플의 4개 특성값은 막대그래프로 그리고 품종을 제목에 표시하세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 4. Iris 모델에 weight decay와 과적합 억제 적용

**선행:** 문제 2

1. 4→12→8→3 구조에 ReLU와 softmax를 사용하세요.
2. AdamW(learning_rate=0.001, weight_decay=0.0001)를 적용하세요.
3. Dropout(0.1)과 검증 손실 기준 EarlyStopping을 사용하세요.
4. one-hot 정답에 맞는 손실을 지정하고 검증 데이터와 함께 학습하세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 5. Iris 학습 곡선과 최종 평가

**선행:** 문제 4

1. 훈련 손실과 검증 손실을 같은 그래프에 그리세요.
2. 테스트 확률에서 argmax로 라벨을 구하고 정확도와 혼동행렬을 출력하세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 6. Pima의 결측값 처리와 표준화

**선행:** 공통 준비

1. Pima CSV의 plasma, pressure, thickness, insulin, bmi에 있는 0을 NaN으로 바꾸세요.
2. pregnant의 0은 유지하고 diabetes를 정답으로 분리하세요.
3. 층화 60/20/20 분할 후 훈련 중앙값으로 대체하고 표준화하세요.


In [ ]:
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/pima-indians-diabetes3.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 pima-indians-diabetes3.csv 위치로 수정하세요."
    )

# 각 문제에서 사용할 원본 CSV를 읽습니다.
pima_df = pd.read_csv(FILE_PATH)

# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 7. Pima 이진 분류 모델과 F1 평가

**선행:** 문제 6

1. 8→16→8→1 구조에서 출력에 sigmoid, 손실에 binary cross entropy를 사용하세요.
2. AdamW와 Dropout 또는 EarlyStopping 등 과적합 억제를 적용하세요.
3. 확률 0.5를 기준으로 테스트 정확도와 F1을 출력하세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 8. Ames의 수치·범주 전처리와 가격 변환

**선행:** 공통 준비

1. OverallQual, GrLivArea, GarageCars, TotalBsmtSF, Neighborhood로 SalePrice를 예측하도록 준비하세요.
2. 60/20/20 분할 후 수치형은 중앙값 대체·표준화, Neighborhood는 one-hot 인코딩하세요.
3. 새 범주를 무시하도록 설정하고, 정답 가격에는 log1p 후 표준화를 적용하세요.


In [ ]:
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/ames_train.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 ames_train.csv 위치로 수정하세요."
    )

# 각 문제에서 사용할 원본 CSV를 읽습니다.
ames_df = pd.read_csv(FILE_PATH)

# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 9. Ames 회귀와 원래 가격 단위의 MAE

**선행:** 문제 8

1. 전처리된 입력→32→16→1의 선형 출력 모델을 MSE로 학습하세요.
2. AdamW와 Dropout·EarlyStopping을 적용하세요.
3. 출력의 표준화와 로그 변환을 역으로 되돌린 뒤 테스트 MAE를 계산하세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 10. ETTh1 시간순 분할과 window 생성

**선행:** 공통 준비

1. 날짜를 변환·정렬하고 OT만 사용하세요. 시간순 70/15/15 경계를 잡으세요.
2. 앞 70% 원시 OT에만 StandardScaler를 fit하세요.
3. 과거 24개→다음 1개 window를 만들고 정답 시점으로 분할하세요.


In [ ]:
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/ETTh1.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 ETTh1.csv 위치로 수정하세요."
    )

# 각 문제에서 사용할 원본 CSV를 읽습니다.
etth_df = pd.read_csv(FILE_PATH)

# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 11. ETTh1 학습과 Last Value 기준선 비교

**선행:** 문제 10

1. 24→32→16→1 구조를 MSE로 학습하고 shuffle=False로 설정하세요.
2. 예측을 원래 OT 단위로 복원하고 테스트 MAE를 출력하세요.
3. 마지막 입력값을 예측으로 쓰는 Last Value와 비교하고 앞 100개를 그리세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 12. IRB2400 FK: 입력·출력 표준화와 다중 회귀

**선행:** 공통 준비

1. q1_in~q6_in으로 x,y,z,yaw,pitch,roll을 예측하도록 64/16/20으로 분할하세요.
2. 입력과 출력에 각각 StandardScaler를 훈련 기준으로 적용하세요.
3. 6→64→32→6 모델에 AdamW와 과적합 억제를 적용하고 출력별 MAE를 계산하세요.
4. 위치 오차(mm)와 자세 오차(rad)를 구분해 출력하세요.


In [ ]:
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/datasetIRB2400.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 datasetIRB2400.csv 위치로 수정하세요."
    )

# 각 문제에서 사용할 원본 CSV를 읽습니다.
fk_df = pd.read_csv(FILE_PATH)

# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 13. IRB2400 IK: 관절각 예측과 주기성 평가

**선행:** 공통 준비

1. 입력 pose 6개, 정답 q1_out~q6_out을 선택하고 64/16/20으로 나누세요.
2. 입력·출력을 각각 표준화하고 6→64→32→6 모델을 학습하세요.
3. 역변환 후 일반 MAE와 atan2(sin(오차),cos(오차)) 기반 각도 MAE를 관절별로 출력하세요.


In [ ]:
FILE_PATH = "/content/drive/MyDrive/Colab Notebooks/data/datasetIRB2400.csv"

if not os.path.exists(FILE_PATH):
    raise FileNotFoundError(
        f"데이터 파일을 찾을 수 없습니다: {FILE_PATH}\n"
        "FILE_PATH를 실제 datasetIRB2400.csv 위치로 수정하세요."
    )

# 각 문제에서 사용할 원본 CSV를 읽습니다.
ik_df = pd.read_csv(FILE_PATH)

# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.


## 문제 14. Iris 모델·전처리 저장과 다시 예측

**선행:** 문제 2, 4

1. 문제 4의 모델을 .keras로 저장하고 scaler와 라벨 인코더를 함께 저장하세요.
2. 원시 테스트 샘플 3개에 저장된 scaler와 재로딩한 모델을 적용하세요.
3. 저장 전후 확률이 같은지 확인하고 예측 품종을 출력하세요.


In [ ]:
# TODO: 위 문제의 요구 사항을 이 셀에 구현하세요.
